# FastFlow на датасете принтера: DeiT Base Distilled

Финальный протокол `printer_384_v2_final`:

- train и normal-loss validation разделены по исходным изображениям;
- calibration и test не пересекаются по source image, object или SHA-256;
- top-k и threshold выбираются только на calibration;
- test используется с зафиксированными параметрами;
- результаты сохраняются в новых каталогах и не перезаписываются по умолчанию.

DeiT использует архитектурно-специфичный режим: 40 эпох, lr=3e-5, flow_steps=8, hidden_ratio=0.5 и gradient clipping 10.0. Clipping применяется только здесь; его фактическая активность записывается в train history.


In [ ]:
# При необходимости установить зависимости в активное окружение:
# %pip install anomalib==2.1.0 torch torchvision scikit-learn pandas tqdm


In [ ]:
import gc
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (cwd, cwd.parent)
    if (candidate / "datasets").is_dir()
    and (candidate / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from fastflow_printer_pipeline import (
    PRINTER_CONFIGS,
    file_sha256,
    load_approved_manifest,
    load_experiment_result,
    run_experiment,
)

DATASET_ROOT = PROJECT_ROOT / "datasets" / "processed_printer_dataset_384"
MANIFEST_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "printer"
    / "dataset_v384_audit"
    / "printer_split_v2_final.csv"
)
SPLIT_CONFIG_PATH = PROJECT_ROOT / "configs" / "printer_split_v2_final.json"
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments" / "printer"


In [ ]:
TRY_NUMBER = 1
SEEDS = [42, 123, 2025]
CONFIG_NAMES = ['deit_base_distilled_384']

# train_calibrate: new training + calibration, without reading test
# test: load existing weights and locked calibration selection, then evaluate test
# load: read test metrics when present, otherwise calibration metrics
EXECUTION_MODE = "train_calibrate"
ALLOW_OVERWRITE = False
DETERMINISTIC = True
BOOTSTRAP_ITERATIONS = 2000

assert EXECUTION_MODE in {"train_calibrate", "test", "load"}
for config_name in CONFIG_NAMES:
    if config_name not in PRINTER_CONFIGS:
        raise KeyError(config_name)


In [ ]:
manifest, split_config = load_approved_manifest(
    DATASET_ROOT,
    MANIFEST_PATH,
    SPLIT_CONFIG_PATH,
    verify_hashes=True,
)

split_summary = (
    manifest.groupby(["split", "class_name"])
    .agg(
        images=("path", "size"),
        source_images=("source_group", "nunique"),
        objects=("object_group", "nunique"),
    )
    .reset_index()
)
print(f"split_version={split_config['version']}")
print(f"manifest_sha256={file_sha256(MANIFEST_PATH)}")
display(split_summary)


In [ ]:
results = []
for seed in SEEDS:
    for config_name in CONFIG_NAMES:
        config = PRINTER_CONFIGS[config_name]
        if EXECUTION_MODE == "load":
            result = load_experiment_result(
                EXPERIMENTS_ROOT,
                config,
                TRY_NUMBER,
                seed,
            )
        else:
            result = run_experiment(
                config=config,
                seed=seed,
                dataset_root=DATASET_ROOT,
                manifest_path=MANIFEST_PATH,
                split_config_path=SPLIT_CONFIG_PATH,
                experiments_root=EXPERIMENTS_ROOT,
                try_number=TRY_NUMBER,
                run_training=EXECUTION_MODE == "train_calibrate",
                run_calibration=EXECUTION_MODE == "train_calibrate",
                run_test=EXECUTION_MODE == "test",
                allow_overwrite=ALLOW_OVERWRITE,
                deterministic=DETERMINISTIC,
                verify_manifest_hashes=False,
                bootstrap_iterations=BOOTSTRAP_ITERATIONS,
            )
        results.append(result)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary_df = pd.DataFrame(results)


In [ ]:
if "test_source_group_balanced_tile_roc_auc" in summary_df:
    summary_stage = "test"
    metric_columns = [
        "test_source_group_balanced_tile_roc_auc",
        "test_primary_roc_auc_ci_low",
        "test_primary_roc_auc_ci_high",
        "test_object_roc_auc",
        "test_source_image_roc_auc",
        "test_source_group_balanced_tile_balanced_accuracy",
        "test_source_group_balanced_tile_fpr",
        "test_source_group_balanced_tile_fnr",
    ]
else:
    summary_stage = "calibration"
    metric_columns = [
        "calibration_source_group_balanced_tile_roc_auc",
        "calibration_object_roc_auc",
        "calibration_source_image_roc_auc",
        "calibration_source_group_balanced_tile_balanced_accuracy",
        "calibration_source_group_balanced_tile_fpr",
        "calibration_source_group_balanced_tile_fnr",
    ]
summary_columns = [
    "config_name",
    "seed",
    "selected_top_k_pixels",
    "selected_top_k_fraction",
    *metric_columns,
    "log_dir",
]
display(summary_df[summary_columns])

if EXECUTION_MODE != "load":
    summary_dir = EXPERIMENTS_ROOT / "printer384_v2_final_summaries"
    summary_path = summary_dir / f"fastflow_deit_printer384_v2_final_{summary_stage}_try_{TRY_NUMBER}.csv"
    if summary_path.exists() and not ALLOW_OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite summary: {summary_path}")
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_path, index=False)
    print(f"summary={summary_path}")


## Интерпретация

Основная ranking-метрика — `test_source_group_balanced_tile_roc_auc`.
Bootstrap-интервал учитывает зависимость тайлов внутри одной исходной съёмки.
Режим `train_calibrate` не читает test. Для финальной оценки нужно отдельно переключить `EXECUTION_MODE` на `test`.
Метрики threshold используют порог, полученный только из normal calibration.
Значения test нельзя использовать для изменения top-k, threshold или параметров обучения.
